# Silver_BIX_LAH_DF2

Reads raw CSVs from **Bronze Lakehouse** → SCD Type 2 → **Silver Lakehouse** Delta tables.

| Step | Descripción |
|---|---|
| **Config** | Lakehouse names + table array |
| **Step 1** | `step1_scd2_pipeline()` — 17 tablas con SCD2 completo |
| **Step 2** | `step2_computed_cols()` — WorkingDays, mrrrMoveOutDate, MRWorkingDays en dimservicerequest |
| **Step 3** | `step3_validate()` — row counts, SCD2 integrity, computed cols, 9 propiedades |

## Config — Lakehouse Names

In [ ]:
# ============================================================
# Silver_BIX_LAH_DF2  —  CONFIG
# ============================================================
# HOW TO USE:
#   1. Attach THIS notebook to Heritage_Silver_LH (default lakehouse)
#   2. Also add Heritage_Bronze_LH as a second lakehouse (read-only)
#   3. Set BRONZE_LH_NAME to the exact name of Bronze lakehouse in Fabric
# ============================================================

BRONZE_LH_NAME = "Heritage_Bronze_LH"   # lakehouse where Bronze writes raw CSVs
SILVER_DB      = spark.sql("SELECT current_database()").collect()[0][0]

RAW_DIR = f"Files/RealPageDaily"         # path inside Bronze lakehouse

print(f"Silver DB   : {SILVER_DB}")
print(f"Bronze LH   : {BRONZE_LH_NAME}")
print(f"Raw CSV dir : {RAW_DIR}")

## Tables Array — 17 BIX tables

In [ ]:
TECH_SCD2_COLS = ["hash_value", "Start_Date", "End_Date", "IsCurrent"]

tables = [
  {"name":"dimaccountgrouphierarchy__0001",       "business_key":["AccountGroupHierarchyKey"],
   "exclude_hash_cols":["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]},
  {"name":"dimclassificationlist__0001",           "business_key":["ClassificationListKey"],
   "exclude_hash_cols":["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]},
  {"name":"dimleadmanagement__0001",               "business_key":["LeadManagementKey"],
   "exclude_hash_cols":["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]},
  {"name":"dimleadmetrics__0001",                  "business_key":["LeadMetricsKey"],
   "exclude_hash_cols":["RecordCreatedDate","RecordModifiedDate","Date","Processed_Time",*TECH_SCD2_COLS]},
  {"name":"dimleaseattributes__0001",              "business_key":["LeaseAttributesKey"],
   "exclude_hash_cols":["RecordCreatedDate","RecordModifiedDate","CDSExtractDate",*TECH_SCD2_COLS]},
  {"name":"dimresidentactivitylog__0001",          "business_key":["ResidentActivityLogkey"],
   "include_hash_cols":["ResidentActivityLogkey","PropertyKey","ResidentKey","CodeActivityTypeCode"]},
  {"name":"dimleaseexpiration__0001",              "business_key":["LeaseExpirationKey"],
   "exclude_hash_cols":["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]},
  {"name":"dimmakereadyrequest__0001",             "business_key":["MakeReadyRequestKey"],
   "exclude_hash_cols":["RecordCreatedDate","RecordModifiedDate","osl_ExtractDate","CreateDate","UnitKey",*TECH_SCD2_COLS]},
  {"name":"dimproperty__0001",                     "business_key":["PropertyKey"],
   "include_hash_cols":["PropertyKey","OrganizationKey","PropertyName","PropertyAddress1"]},
  {"name":"dimresidentmember__0001",               "business_key":["ResidentMemberKey"],
   "exclude_hash_cols":["RecordCreatedDate","RecordModifiedDate","CDSExtractDate",*TECH_SCD2_COLS]},
  {"name":"dimscreeningapplicant__0001",           "business_key":["ApplicantKey"],
   "exclude_hash_cols":["RecordCreatedDate","RecordModifiedDate","RowStartDate","BirthDate","ApplicationDate",*TECH_SCD2_COLS]},
  {"name":"dimservicerequest__0001",               "business_key":["ServiceRequestKey"],
   "exclude_hash_cols":["RecordCreatedDate","RecordModifiedDate","RequestTime","CompleteTime","UnitKey",
                        "WorkingDays","mrrrMakeReadyStartDay","MRWorkingDays","mrrrMoveOutDate",*TECH_SCD2_COLS]},
  {"name":"factoperationalkpi__0001",              "business_key":["PropertyKey"],
   "exclude_hash_cols":["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]},
  {"name":"factaccountgrouphierarchydetail__0001", "business_key":["AccountGroupHierarchyDetailKey"],
   "exclude_hash_cols":["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]},
  {"name":"factglsummary__0001",                   "business_key":["Tablekey"],
   "exclude_hash_cols":["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]},
  {"name":"factpropertystatisticsasofdate__0001",  "business_key":["PropertyKey","PostDate"],
   "exclude_hash_cols":["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]},
  {"name":"factscreeningapplicant__0001",          "business_key":["ApplicantKey"],
   "exclude_hash_cols":["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]},
]

## SCD2 Engine — Shared Functions

In [ ]:
# ============================================================
# SCD2 ENGINE — shared helpers
# ============================================================
import re
from datetime import date, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DateType, TimestampType
from pyspark.sql.window import Window
from notebookutils import mssparkutils

HASH_COL      = "hash_value"
START_COL     = "Start_Date"
END_COL       = "End_Date"
ISCURRENT_COL = "IsCurrent"
MAX_END       = "12/31/9999"
DEFAULT_START = "1/1/2025"

spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

DATE_PATTERNS = ["M/d/yyyy","MM/dd/yyyy","M/d/yy","MM/dd/yy","yyyy-MM-dd","yyyy/MM/dd","yyyyMMdd",
                 "dd-MMM-yyyy","d-MMM-yyyy","MMM d, yyyy","MMMM d, yyyy"]
TS_PATTERNS   = ["M/d/yyyy h:mm:ss a","MM/dd/yyyy h:mm:ss a","M/d/yyyy H:mm:ss","MM/dd/yyyy H:mm:ss",
                 "M/d/yyyy HH:mm:ss","MM/dd/yyyy HH:mm:ss","yyyy-MM-dd HH:mm:ss",
                 "yyyy-MM-dd'T'HH:mm:ss","yyyy-MM-dd'T'HH:mm:ss.SSS"]

def _pdate(col_trim):
    ts = F.coalesce(*[F.to_timestamp(col_trim, p) for p in TS_PATTERNS])
    d  = F.coalesce(*[F.to_date(col_trim, p) for p in DATE_PATTERNS])
    return F.coalesce(F.to_date(ts), d)

def standardize_dates(df, sample_rows=1000, threshold=0.90, min_nn=10):
    out = df
    for fld in df.schema.fields:
        c, dtp = fld.name, fld.dataType
        if isinstance(dtp, (DateType, TimestampType)):
            out = out.withColumn(c, F.date_format(F.to_date(F.col(c)), "M/dd/yyyy"))
        elif isinstance(dtp, StringType):
            ct = F.trim(F.col(c))
            pd = _pdate(ct)
            s  = df.limit(sample_rows).select(
                F.sum(F.when(ct.isNotNull() & (F.length(ct)>0), 1).otherwise(0)).alias("nn"),
                F.sum(F.when((ct.isNotNull() & (F.length(ct)>0)) & pd.isNotNull(), 1).otherwise(0)).alias("ok")
            ).collect()[0]
            nn, ok = int(s["nn"] or 0), int(s["ok"] or 0)
            if nn >= min_nn and (ok/nn if nn else 0) >= threshold:
                out = out.withColumn(c, F.date_format(pd, "M/dd/yyyy"))
    return out

def _nk(s): return re.sub(r"[^a-z0-9]+","", (s or "").lower().strip())

def norm_excl(df, cols):
    m = {_nk(c): c for c in df.columns}
    return [m[_nk(str(x))] for x in (cols or []) if _nk(str(x)) in m]

def add_hash(df, exclude_cols=None, include_cols=None):
    if include_cols:
        cols = sorted([c for c in df.columns if c in set(include_cols)], key=str.lower)
    else:
        ex   = set((exclude_cols or []) + [HASH_COL,START_COL,END_COL,ISCURRENT_COL])
        cols = sorted([c for c in df.columns if c not in ex], key=str.lower)
    if not cols: return df.withColumn(HASH_COL, F.sha2(F.lit(""),256))
    return df.withColumn(HASH_COL, F.sha2(F.concat_ws("||",*[F.coalesce(F.trim(F.col(c).cast("string")),F.lit("")) for c in cols]),256))

def _resolve_hash(df, cfg):
    incl = cfg.get("include_hash_cols")
    if incl: return {"include_cols": norm_excl(df, incl)}
    return {"exclude_cols": norm_excl(df, cfg.get("exclude_hash_cols",[]))}

def _ots(df, c):
    if c not in df.columns: return None
    return F.to_timestamp(_pdate(F.trim(F.col(c).cast("string"))))

def dedup_latest(df, bk):
    oc = ["RecordModifiedDate","RecordCreatedDate","CDSExtractDate","osl_ExtractDate",
          "CreateDate","RowStartDate","ModifyDate","PostDate"]
    oe = [ts.desc_nulls_last() for c in oc for ts in [_ots(df,c)] if ts]
    if HASH_COL in df.columns: oe.append(F.col(HASH_COL).desc_nulls_last())
    for c in bk:
        if c in df.columns: oe.append(F.col(c).cast("string").desc_nulls_last())
    if not oe: return df.dropDuplicates(bk)
    w = Window.partitionBy(*[F.col(c) for c in bk]).orderBy(*oe)
    return df.withColumn("_rn",F.row_number().over(w)).where(F.col("_rn")==1).drop("_rn")

def ensure_tech(df):
    if START_COL     not in df.columns: df = df.withColumn(START_COL,     F.lit(DEFAULT_START))
    if END_COL       not in df.columns: df = df.withColumn(END_COL,       F.lit(MAX_END))
    if ISCURRENT_COL not in df.columns: df = df.withColumn(ISCURRENT_COL, F.lit(1))
    return df

def _has(df): return df.limit(1).count() > 0

def consolidate_ranges(df, bk):
    sp = _pdate(F.trim(F.col(START_COL).cast("string")))
    ep = F.when(F.trim(F.col(END_COL))==F.lit(MAX_END), F.to_date(F.lit(MAX_END),"M/d/yyyy")).otherwise(_pdate(F.trim(F.col(END_COL).cast("string"))))
    grp = bk+[HASH_COL]
    w   = Window.partitionBy(*[F.col(c) for c in grp])
    wf  = Window.partitionBy(*[F.col(c) for c in grp]).orderBy(sp.asc_nulls_last())
    df  = df.withColumn("_sp",sp).withColumn("_ep",ep)
    df  = df.withColumn("_smin",F.min("_sp").over(w)).withColumn("_emax",F.max("_ep").over(w))
    df  = df.withColumn("_rn",F.row_number().over(wf)).where(F.col("_rn")==1).drop("_rn")
    df  = df.withColumn(START_COL, F.date_format("_smin","M/dd/yyyy"))
    df  = df.withColumn(END_COL, F.when(F.col("_emax")==F.to_date(F.lit(MAX_END),"M/d/yyyy"),F.lit(MAX_END)).otherwise(F.date_format("_emax","M/dd/yyyy")))
    sp2 = _pdate(F.trim(F.col(START_COL).cast("string")))
    ep2 = F.when(F.trim(F.col(END_COL))==F.lit(MAX_END),F.to_date(F.lit(MAX_END),"M/d/yyyy")).otherwise(_pdate(F.trim(F.col(END_COL).cast("string"))))
    wbk = Window.partitionBy(*[F.col(c) for c in bk]).orderBy(sp2.asc_nulls_last())
    ns  = F.lead(sp2).over(wbk)
    ef  = F.when((F.trim(F.col(END_COL))==F.lit(MAX_END))&ns.isNotNull(), F.date_sub(ns,1)).otherwise(ep2)
    ef  = F.greatest(sp2, ef)
    df  = df.withColumn("_ef",ef).withColumn(END_COL,
        F.when(F.col("_ef")==F.to_date(F.lit(MAX_END),"M/d/yyyy"),F.lit(MAX_END)).otherwise(F.date_format("_ef","M/dd/yyyy")))
    df  = df.drop("_sp","_ep","_smin","_emax","_ef")
    return df.withColumn(ISCURRENT_COL, F.when(F.trim(F.col(END_COL))==F.lit(MAX_END),F.lit(1)).otherwise(F.lit(0)))

def force_single_current(df, bk):
    ed = F.when(F.trim(F.col(END_COL))==F.lit(MAX_END),F.to_date(F.lit(MAX_END),"M/d/yyyy")).otherwise(_pdate(F.trim(F.col(END_COL).cast("string"))))
    sd = _pdate(F.trim(F.col(START_COL).cast("string")))
    w  = Window.partitionBy(*[F.col(c) for c in bk]).orderBy(ed.desc_nulls_last(),sd.desc_nulls_last())
    return df.withColumn("_rc",F.row_number().over(w))\
             .withColumn(END_COL, F.when(F.col("_rc")==1,F.lit(MAX_END)).otherwise(F.col(END_COL)))\
             .withColumn(ISCURRENT_COL, F.when(F.col("_rc")==1,F.lit(1)).otherwise(F.lit(0))).drop("_rc")

def _pick_csv(arr_name):
    want  = _nk(arr_name)
    items = mssparkutils.fs.ls(RAW_DIR)
    for it in items:
        fname = getattr(it,"name",None) or it.path.split("/")[-1]
        stem  = re.sub(r"\.csv$","",fname,flags=re.I)
        if _nk(stem) == want: return getattr(it,"path",f"{RAW_DIR}/{fname}")
    raise ValueError(f"No CSV for {arr_name} in {RAW_DIR}")

def read_raw(name):
    path = _pick_csv(name)
    return spark.read.option("header",True).option("inferSchema",True).csv(path)

def load_parent(parent, ref_df):
    try:    return spark.table(parent)
    except: return spark.createDataFrame([], ref_df.schema)

print("SCD2 engine loaded.")

## Step 1 — SCD2 Pipeline

In [ ]:
# ============================================================
# STEP 1 — CSV (Bronze) → STG → SCD2 Delta Tables (Silver)
# ============================================================
def step1_scd2_pipeline():
    TODAY     = date.today()
    YESTERDAY = TODAY - timedelta(days=1)
    TODAY_STR = f"{TODAY.month}/{TODAY.day}/{TODAY.year}"
    YEST_STR  = f"{YESTERDAY.month}/{YESTERDAY.day}/{YESTERDAY.year}"

    failed = []
    for cfg in tables:
        name   = cfg["name"]
        parent = name
        child  = f"stg_{name}"
        bk     = cfg.get("business_key",[])

        try:
            print(f"\n=== {name} ===")
            if not bk: raise ValueError("empty business_key")

            # RAW → STG with hash
            df_raw = read_raw(name)
            df_raw = standardize_dates(df_raw)
            df_raw = ensure_tech(df_raw)
            df_stg = add_hash(df_raw, **_resolve_hash(df_raw, cfg))
            df_stg.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(child)
            stg_count = df_stg.count()
            print(f"  STG: {child} | rows={stg_count}")

            # SCD2 merge
            df_child  = spark.table(child)
            df_parent = load_parent(parent, df_child)
            df_parent = standardize_dates(df_parent)
            df_parent = ensure_tech(df_parent)
            df_parent = add_hash(df_parent, **_resolve_hash(df_parent, cfg))
            df_child  = dedup_latest(df_child, bk).cache()
            _ = df_child.count()

            c  = df_child.alias("c")
            pc = df_parent.where(F.trim(F.col(END_COL))==F.lit(MAX_END)).alias("pc")
            ph = df_parent.where(F.trim(F.col(END_COL))!=F.lit(MAX_END)).alias("ph")
            cond_cp = [F.col(f"c.{x}").cast("string")==F.col(f"pc.{x}").cast("string") for x in bk]

            changed = (c.join(pc, on=cond_cp, how="inner")
                .where(F.col(f"c.{HASH_COL}")!=F.col(f"pc.{HASH_COL}"))
                .select([F.col(f"c.{x}").alias(x) for x in bk]).dropDuplicates(bk))
            new_bk  = c.join(pc, on=cond_cp, how="left_anti").select("c.*").dropDuplicates(bk)

            still_ok = pc.select("pc.*")
            to_close = None
            if _has(changed):
                k = changed.alias("k")
                cond_pk = [F.col(f"pc.{x}").cast("string")==F.col(f"k.{x}").cast("string") for x in bk]
                sp = _pdate(F.trim(F.col(f"pc.{START_COL}").cast("string")))
                safe_end = F.least(F.to_date(F.lit(TODAY_STR),"M/d/yyyy"),
                                   F.greatest(sp, F.to_date(F.lit(YEST_STR),"M/d/yyyy")))
                to_close  = pc.join(k, on=cond_pk, how="inner").select("pc.*").withColumn(END_COL, F.date_format(safe_end,"M/dd/yyyy"))
                still_ok  = pc.join(k, on=cond_pk, how="left_anti").select("pc.*")

            df_out = ph.unionByName(still_ok, allowMissingColumns=True)
            if to_close is not None:
                df_out = df_out.unionByName(to_close, allowMissingColumns=True)
            if _has(new_bk):
                df_out = df_out.unionByName(new_bk.withColumn(START_COL,F.lit(TODAY_STR)).withColumn(END_COL,F.lit(MAX_END)), allowMissingColumns=True)
            if _has(changed):
                k2 = changed.alias("k2")
                ck = [F.col(f"c.{x}").cast("string")==F.col(f"k2.{x}").cast("string") for x in bk]
                cn = c.join(k2,on=ck,how="inner").select("c.*").dropDuplicates(bk)
                if _has(cn):
                    df_out = df_out.unionByName(cn.withColumn(START_COL,F.lit(TODAY_STR)).withColumn(END_COL,F.lit(MAX_END)), allowMissingColumns=True)

            df_out = standardize_dates(df_out)
            df_out = consolidate_ranges(df_out, bk)
            df_out = force_single_current(df_out, bk)
            df_out.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(parent)
            df_child.unpersist()
            print(f"  SCD2 OK | final rows={df_out.count()}")

        except Exception as e:
            failed.append((name, str(e)[:300]))
            import traceback; traceback.print_exc()

    if failed:
        raise Exception(f"SCD2 failed on {len(failed)} tables: {[n for n,_ in failed]}")
    print(f"\nStep 1 complete: {len(tables)} tables processed.")

## Step 2 — dimservicerequest Computed Columns

Equivalente PySpark de las columnas DAX: `WorkingDays`, `mrrrMoveOutDate`, `MRWorkingDays`

In [ ]:
# ============================================================
# STEP 2 — dimservicerequest__0001: Computed Columns
#
# WorkingDays    : business days CreateDate  → ActualCompletedDate (or TODAY)
# mrrrMoveOutDate: LOOKUPVALUE dimmakereadyrequest where TypeCode = "MR"
# MRWorkingDays  : business days mrrrMoveOutDate → ActualCompletedDate (MR only)
# ============================================================
def _pdate_sr(col_):
    s = F.trim(col_.cast("string"))
    sn = F.when(s.rlike(r"^\d{1,2}/\d{1,2}/\d{4}$"),
        F.concat_ws("/", F.lpad(F.split(s,"/").getItem(0),2,"0"),
                         F.lpad(F.split(s,"/").getItem(1),2,"0"),
                         F.split(s,"/").getItem(2))
    ).otherwise(s)
    return F.coalesce(F.to_date(s,"yyyy-MM-dd"), F.to_date(s,"yyyy-MM-dd HH:mm:ss"),
                      F.to_date(s,"yyyy-MM-dd'T'HH:mm:ss"), F.to_date(sn,"MM/dd/yyyy"), F.to_date(sn,"MM/dd/yyyy HH:mm:ss"))

def _build_cal(df_date):
    cal = (df_date.select(F.col("Date").cast("date").alias("CalDate"), F.col("NonWorkDays").cast("int").alias("NWD"))
        .filter(F.col("CalDate").isNotNull()).dropDuplicates(["CalDate"])
        .withColumn("IsWork", F.when(F.col("NWD")==0,F.lit(1)).otherwise(F.lit(0))))
    w   = Window.orderBy("CalDate").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    cal = cal.withColumn("Cum", F.sum("IsWork").over(w)).withColumn("Before", F.col("Cum")-F.col("IsWork")).select("CalDate","Cum","Before")
    cs  = cal.select(F.col("CalDate").alias("_SC"), F.col("Before").alias("_BS"))
    ce  = cal.select(F.col("CalDate").alias("_EC"), F.col("Cum").alias("_CE"))
    return cs, ce

def _wd(df, sc, ec, oc, cs, ce, sa, ea):
    csa = cs.select(F.col("_SC").alias(sa+"c"), F.col("_BS").alias(sa+"b"))
    cea = ce.select(F.col("_EC").alias(ea+"c"), F.col("_CE").alias(ea+"e"))
    df  = df.join(csa, df[sc]==csa[sa+"c"], "left").join(cea, df[ec]==cea[ea+"c"], "left")
    df  = df.withColumn(oc,
        F.when(F.col(sc).isNull()|F.col(ec).isNull()|F.col(sa+"b").isNull()|F.col(ea+"e").isNull()|(F.col(ec)<F.col(sc)), F.lit(None).cast("int"))
         .otherwise(F.greatest((F.col(ea+"e")-F.col(sa+"b"))-F.lit(1), F.lit(0)).cast("int")))
    return df.drop(sa+"c",sa+"b",ea+"c",ea+"e")

def step2_computed_cols():
    sr   = spark.table("dimservicerequest__0001")
    dt   = spark.table("Date")
    mr   = (spark.table("dimmakereadyrequest__0001")
            .select("ServiceRequestKey",
                    F.col("mrrMakeReadyStartDay").alias("_ms"),
                    F.col("mrrMoveOutDate").alias("_mo"))
            .dropDuplicates(["ServiceRequestKey"]))

    cs, ce = _build_cal(dt)

    sr = sr.withColumn("_ED", F.coalesce(_pdate_sr(F.col("ActualCompletedDate")), F.current_date()))
    sr = sr.withColumn("_SD", _pdate_sr(F.col("CreateDate")))
    sr = _wd(sr, "_SD", "_ED", "WorkingDays", cs, ce, "wd_s", "wd_e")

    sr = (sr.join(mr, on="ServiceRequestKey", how="left")
            .withColumn("mrrrMakeReadyStartDay",
                F.when(F.upper(F.trim(F.col("ServiceRequestTypeCode")))==F.lit("MR"), F.col("_ms")).otherwise(F.lit(None)))
            .withColumn("mrrrMoveOutDate",
                F.when(F.upper(F.trim(F.col("ServiceRequestTypeCode")))==F.lit("MR"), F.col("_mo")).otherwise(F.lit(None)))
            .drop("_ms","_mo"))

    sr = sr.withColumn("_MRS", _pdate_sr(F.col("mrrrMoveOutDate")))
    sr = _wd(sr, "_MRS", "_ED", "MRWorkingDays", cs, ce, "mr_s", "mr_e")
    sr = sr.drop("_SD","_ED","_MRS")

    cnt = sr.count()
    sr.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("dimservicerequest__0001")
    print(f"Step 2 OK: dimservicerequest__0001 | rows={cnt}")
    sr.select("ServiceRequestKey","ServiceRequestTypeCode","WorkingDays","mrrrMoveOutDate","MRWorkingDays").show(5, truncate=False)

## Step 3 — Data Validation

Verifica integridad antes de continuar al Gold layer.

In [ ]:
# ============================================================
# STEP 3 — DATA VALIDATION
# Stops the pipeline if critical checks fail
# ============================================================
def step3_validate():
    from pyspark.sql import functions as F

    checks = []

    def chk(name, df, condition_expr, expect_zero=True):
        cnt = df.filter(condition_expr).count()
        status = ("PASS" if cnt == 0 else "FAIL") if expect_zero else ("PASS" if cnt > 0 else "FAIL")
        checks.append({"check": name, "count": cnt, "status": status})
        icon = "OK" if status == "PASS" else "FAIL"
        print(f"  [{icon}] {name}: {cnt}")

    print("=== Validation: Row counts ===")
    for tbl in [t["name"] for t in tables]:
        try:
            cnt = spark.table(tbl).count()
            ok  = cnt > 0
            checks.append({"check": f"{tbl} rows>0", "count": cnt, "status": "PASS" if ok else "FAIL"})
            print(f"  {'OK' if ok else 'FAIL'} {tbl}: {cnt:,} rows")
        except Exception as e:
            checks.append({"check": f"{tbl} rows>0", "count": -1, "status": "FAIL"})
            print(f"  FAIL {tbl}: {e}")

    print("\n=== Validation: SCD2 integrity — no duplicate IsCurrent=1 per BK ===")
    for cfg in tables:
        tbl, bk = cfg["name"], cfg.get("business_key",[])
        if not bk: continue
        try:
            df = spark.table(tbl)
            curr_df = df.filter(F.col("IsCurrent")==1)
            dupes   = curr_df.groupBy(*bk).count().filter(F.col("count")>1).count()
            ok = dupes == 0
            checks.append({"check": f"{tbl} no_dup_current", "count": dupes, "status": "PASS" if ok else "FAIL"})
            print(f"  {'OK' if ok else 'FAIL'} {tbl}: {dupes} duplicate current keys")
        except Exception as e:
            checks.append({"check": f"{tbl} no_dup_current", "count": -1, "status": "FAIL"})

    print("\n=== Validation: dimservicerequest computed cols ===")
    sr = spark.table("dimservicerequest__0001")
    chk("WorkingDays no negative",    sr, F.col("WorkingDays") < 0)
    chk("MRWorkingDays no negative",  sr, F.col("MRWorkingDays") < 0)
    chk("WorkingDays column exists",  sr.filter(F.col("WorkingDays").isNotNull()), F.lit(True)==F.lit(True), expect_zero=False)

    print("\n=== Validation: dimproperty__0001 — at least 9 properties ===")
    prop_cnt = spark.table("dimproperty__0001").filter(F.col("IsCurrent")==1).select("PropertyKey").distinct().count()
    ok = prop_cnt >= 9
    checks.append({"check":"properties>=9","count":prop_cnt,"status":"PASS" if ok else "FAIL"})
    print(f"  {'OK' if ok else 'FAIL'} Active properties: {prop_cnt}")

    # Summary
    failed = [c for c in checks if c["status"]=="FAIL"]
    passed = [c for c in checks if c["status"]=="PASS"]
    print(f"\nValidation Summary: {len(passed)} PASS | {len(failed)} FAIL")

    if failed:
        for f in failed:
            print(f"  FAIL: {f['check']} (count={f['count']})")
        raise Exception(f"Validation failed: {len(failed)} checks. First: {failed[0]['check']}")

    print("All validations passed.")

## Pipeline Runner

In [ ]:
# ============================================================
# PIPELINE RUNNER — Silver BIX
# ============================================================
import traceback

_RESULTS = []

def _run(name, fn):
    print(f"\n{'='*55}")
    print(f"START: {name}")
    print("="*55)
    try:
        fn()
        _RESULTS.append((name,"OK"))
        print(f"DONE: {name}")
    except Exception as e:
        _RESULTS.append((name, str(e)[:300]))
        print(f"FAIL: {name}: {e}")
        print(traceback.format_exc())

_run("Step 1 — SCD2 Pipeline (17 tables)",           step1_scd2_pipeline)
_run("Step 2 — DimServiceRequest Computed Cols",      step2_computed_cols)
_run("Step 3 — Data Validation",                      step3_validate)

print("\n" + "="*55)
print("SILVER BIX PIPELINE SUMMARY")
print("="*55)
for name, status in _RESULTS:
    icon = "OK" if status == "OK" else "FAIL"
    print(f"  [{icon}] {name}" + (f": {status}" if status != "OK" else ""))

if any(s != "OK" for _,s in _RESULTS):
    raise Exception("Silver BIX pipeline had failures.")
print("\nSilver BIX complete.")